# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fardinhossain/FlyRankAI-ML-Internship-Task/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook executes the feature vector construction and rigorous leakage/privacy audit for **Lane 2 — Refresh / Content Opportunity Scoring**.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We load the mature content slice and assemble 19 numeric features and 8 one-hot encoded categorical features, ensuring zero leakage of target derivatives.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Load dataset
candidates = ["data/raw/content_refresh_anonymized.csv", "../../data/raw/content_refresh_anonymized.csv"]
path = next((p for p in candidates if os.path.exists(p)), None)
df = pd.read_csv(path, encoding="utf-8")

# Mature slice filter
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Volume transformations
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])

sys.path.append(os.getcwd())
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

num_feats = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
cat_feats = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

X_num = df[num_feats].fillna(0)
X_cat = pd.get_dummies(df[cat_feats].astype(str), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)

print(f"Feature vector built: {X.shape[0]:,} rows × {X.shape[1]} columns.")
print("Feature columns preview:", list(X.columns[:10]))


Feature vector built: 30,000 rows × 40 columns.
Feature columns preview: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'days_with_impressions', 'days_with_sessions']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every feature represents observable measurements up to decision time $t=0$:
- `log_impressions_90d`: Log-transformed trailing search impressions. Missing values filled with 0. Known at $t=0$.
- `avg_position`: Mean SERP ranking. Values $\le 0$ treated as unranked. Known at $t=0$.
- `content_age_days`: Days since first publication. Fully populated. Known at $t=0$.
- `days_since_last_update`: Days since last editorial modification. Known at $t=0$.
- `word_count` / `char_count`: Text length metrics. Missing filled with 0. Known at $t=0$.
- `content_type` / `competition_level`: High-level taxonomy tiers, one-hot encoded. Known at $t=0$.

In [2]:
# Feature missingness audit
missing_summary = X.isnull().sum()
print("Missing values in feature matrix X:")
print(f"Total nulls: {missing_summary.sum()}")
print("Feature value ranges:")
print(X[num_feats[:5]].describe().round(2).to_string())


Missing values in feature matrix X:
Total nulls: 0
Feature value ranges:
       search_volume  competition       cpc  word_count  char_count
count       30000.00     30000.00  30000.00    30000.00    30000.00
mean          145.81         0.13      0.45     2310.21    15361.88
std          1455.13         0.28      2.02     1846.79    12551.23
min             0.00         0.00      0.00        0.00        0.00
25%             0.00         0.00      0.00        0.00        0.00
50%            10.00         0.00      0.00     2605.00    16984.50
75%            20.00         0.10      0.00     3247.00    21574.00
max         74000.00         1.00    100.36     9546.00   111158.00


## 3. Two leakage / privacy checks

*Audit features for target leakage and verify that no private identifiers appear.*

1. **Target Leakage Check:** Ensure neither `trend_direction` nor `trend_pct` appear in feature matrix $X$.
2. **Privacy Audit:** Ensure neither raw URLs, customer client names, nor unhashed search queries exist in $X$.

In [3]:
# Leakage and Privacy Audit Checks
forbidden = ["trend_direction", "trend_pct", "is_declining_label", "client_name", "url", "query_text"]
found_leaks = [c for c in forbidden if c in X.columns]
print(f"Forbidden columns in feature matrix X: {found_leaks} (Clean: {len(found_leaks) == 0})")

# Check correlation with target
corrs = X.apply(lambda col: col.corr(df['is_declining_label'])).abs().sort_values(ascending=False).head(5)
print("\nTop 5 correlations with label:")
print(corrs.round(4).to_string())
assert corrs.max() < 0.50, "Leaky feature tower detected!"
print("\nLeakage audit passed successfully!")


Forbidden columns in feature matrix X: [] (Clean: True)

Top 5 correlations with label:
days_with_impressions          0.1901
log_impressions_90d            0.1775
position_tier_top_3            0.1751
content_age_days               0.1639
content_type_feedly article    0.1405

Leakage audit passed successfully!


## Self-check

- [x] Feature vector constructed with engineered volume signals and one-hot encoding
- [x] Feature notes document temporal availability and missingness handling
- [x] Leakage audit confirms target derivatives excluded
- [x] Privacy audit confirms no client names or URLs present